# Cloud runner for the frozen NEU-CLS experiment

This notebook orchestrates the registered project on a **Google Colab or Kaggle GPU runtime**. It never embeds credentials and contains no precomputed or fabricated results. Every output must be produced by the checked-out code and downloaded data.

Safety boundaries:

- the committed `data/manifests/neu_cls_v2.csv` is verified, never regenerated;
- official data download requires two explicit acknowledgements;
- training, validation evaluation, freeze creation, and locked-test evaluation are opt-in;
- the locked test cannot run without a locally created human freeze record; and
- generated artifacts can be redirected to mounted persistent storage.


## 1. Open an existing checkout or clone the repository

Use a public checkout or attach the repository to the runtime yourself. Do not paste access tokens into this notebook. For a final registered run, replace `main` with the intended immutable release branch before running the cell. The exact commit SHA is recorded below.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/kujifined/Introduction-to-cv.git"
REPO_REF = "main"
PERSISTENT_ROOT_TEXT = ""  # Example: /content/drive/MyDrive/icv-neu-cls
REGISTERED_CONFIGURATIONS = (
    "resnet18_minimal",
    "resnet18_camera",
    "efficientnet_b0_minimal",
    "efficientnet_b0_camera",
)
REGISTERED_SEEDS = (42, 43, 44)


def find_checkout(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "pyproject.toml").is_file():
            return candidate
    return None


existing_checkout = find_checkout(Path.cwd().resolve())
read_only_kaggle_input = existing_checkout is not None and str(existing_checkout).startswith(
    "/kaggle/input/"
)
if existing_checkout is not None and not read_only_kaggle_input:
    PROJECT_ROOT = existing_checkout
else:
    cloud_root = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
    clone_root = cloud_root if cloud_root.is_dir() else Path.cwd()
    repo_dir = clone_root / "Introduction-to-cv"
    if repo_dir.exists() and not (repo_dir / ".git").exists():
        raise FileExistsError(f"Clone destination is occupied by a non-repository: {repo_dir}")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    PROJECT_ROOT = repo_dir

os.chdir(PROJECT_ROOT)
REVISION = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git revision: {REVISION}")


def run(command: list[str]) -> None:
    print(f"$ {shlex.join(command)}", flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 2. Install the locked environment

The project environment comes from `uv.lock`. The `data` extra supplies the official downloader; the `notebook` extra keeps the checkout usable for the EDA notebook. This cell does not train a model.


In [ ]:
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "uv"],
        check=True,
    )
run(["uv", "sync", "--locked", "--extra", "data", "--extra", "notebook"])

rar_extractors = ("unar", "7z", "unrar")
if not any(shutil.which(name) for name in rar_extractors):
    if shutil.which("apt-get") is None:
        raise RuntimeError("Install unar, 7z, or unrar before downloading NEU-CLS.")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=True)

## 3. Optional persistent artifact storage

Leave `PERSISTENT_ROOT_TEXT` empty for ephemeral runtime storage. In Colab, mount Google Drive yourself first and point it to a directory below `/content/drive/MyDrive`. In Kaggle, use a writable directory below `/kaggle/working` and save the notebook version when finished.

The cell refuses to replace a real local output directory. It only creates new symlinks or reuses symlinks already pointing to the requested location.


In [ ]:
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
GENERATED_REPORTS_ROOT = PROJECT_ROOT / "reports" / "tables" / "generated"


def link_output_directory(local_path: Path, persistent_path: Path) -> None:
    persistent_path.mkdir(parents=True, exist_ok=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve() != persistent_path.resolve():
            raise RuntimeError(f"Existing symlink points elsewhere: {local_path}")
        return
    if local_path.exists():
        raise FileExistsError(f"Refusing to replace existing local artifacts at {local_path}")
    local_path.symlink_to(persistent_path, target_is_directory=True)


if PERSISTENT_ROOT_TEXT.strip():
    persistent_root = Path(PERSISTENT_ROOT_TEXT).expanduser()
    if not persistent_root.parent.exists():
        raise FileNotFoundError("Persistent parent is unavailable; mount storage first.")
    persistent_root.mkdir(parents=True, exist_ok=True)
    link_output_directory(OUTPUTS_ROOT, persistent_root / "outputs")
    link_output_directory(GENERATED_REPORTS_ROOT, persistent_root / "reports_generated")
    print(f"Persistent artifact root: {persistent_root.resolve()}")
else:
    print(f"Ephemeral artifact root: {OUTPUTS_ROOT}")

## 4. Acquire the official NEU-CLS dataset — explicit opt-in

First review the [official NEU dataset page](https://faculty.neu.edu.cn/songkechen/zh_CN/zdylm/263270/list/index.htm). The repository does not redistribute the images and the official page does not state a standard machine-readable licence. Set **both** flags to `True` only after reviewing the source terms. If the dataset is already present at `data/raw/NEU-CLS`, leave this cell disabled and continue to verification.


In [ ]:
DOWNLOAD_OFFICIAL_DATA = False
ACKNOWLEDGE_SOURCE_TERMS = False

if DOWNLOAD_OFFICIAL_DATA:
    if not ACKNOWLEDGE_SOURCE_TERMS:
        raise PermissionError("Review and acknowledge the official source terms first.")
    run(
        [
            "uv",
            "run",
            "python",
            "scripts/download_neu_cls.py",
            "--acknowledge-source-terms",
            "--extract",
        ]
    )
else:
    print("Download disabled. Enable both flags only after reviewing the source terms.")

## 5. Verify the committed v2 authority and every image hash

This cell proves that all four v2 manifest artifacts are tracked and unchanged from `HEAD`, prints the actual manifest SHA-256, and recomputes the SHA-256 of every referenced image. It deliberately contains no preparation or split-generation command. A failure must be investigated; do not regenerate `neu_cls_v2.csv`.


In [ ]:
FROZEN_MANIFEST_ARTIFACTS = (
    Path("data/manifests/neu_cls_v2.csv"),
    Path("data/manifests/neu_cls_v2.audit.json"),
    Path("data/manifests/neu_cls_v2_near_duplicates.csv"),
    Path("data/manifests/neu_cls_reviewed_groups_v2.csv"),
)
for relative_path in FROZEN_MANIFEST_ARTIFACTS:
    tracked = subprocess.run(
        ["git", "ls-files", "--error-unmatch", relative_path.as_posix()],
        cwd=PROJECT_ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if tracked.returncode != 0:
        raise RuntimeError(f"Frozen artifact is not tracked: {relative_path}")
    unchanged = subprocess.run(
        ["git", "diff", "--quiet", "HEAD", "--", relative_path.as_posix()],
        cwd=PROJECT_ROOT,
    )
    if unchanged.returncode != 0:
        raise RuntimeError(f"Frozen artifact differs from HEAD: {relative_path}")

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "neu_cls_v2.csv"
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "NEU-CLS"
MANIFEST_SHA256 = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
print(f"Committed manifest SHA-256: {MANIFEST_SHA256}")
verification_code = (
    "import sys; "
    "from industrial_surface_defects.data import verify_manifest_files; "
    "verify_manifest_files(sys.argv[1], sys.argv[2]); "
    "print('manifest and raw files: OK')"
)
run(
    [
        "uv",
        "run",
        "python",
        "-c",
        verification_code,
        str(MANIFEST_PATH),
        str(DATA_ROOT),
    ]
)

## 6. Inspect CUDA and dry-run the frozen 4 × 3 matrix

The dry-run is safe: it loads configurations and prints all 12 planned destinations without fitting models. The CUDA probe reports the actual runtime state.


In [ ]:
cuda_probe_code = (
    "import torch; "
    "print('torch', torch.__version__); "
    "print('cuda_available', torch.cuda.is_available()); "
    "print('torch_cuda', torch.version.cuda); "
    "print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
)
run(["uv", "run", "python", "-c", cuda_probe_code])
run(["uv", "run", "python", "scripts/run_matrix.py", "--device", "cuda", "--dry-run"])


def require_cuda() -> None:
    requirement_code = (
        "import torch; "
        "assert torch.cuda.is_available(), 'CUDA runtime is required for registered runs'; "
        "print('Using CUDA device:', torch.cuda.get_device_name(0))"
    )
    run(["uv", "run", "python", "-c", requirement_code])

## 7. Train all four configurations × three seeds — long, explicit opt-in

This is the main GPU workload. Review the dry-run first, then change `RUN_REGISTERED_TRAINING` to `True`. `--skip-existing` safely resumes a partially completed matrix without replacing populated run directories. Do not add a suffix to registered runs.


In [ ]:
RUN_REGISTERED_TRAINING = False

if RUN_REGISTERED_TRAINING:
    require_cuda()
    run(
        [
            "uv",
            "run",
            "python",
            "scripts/run_matrix.py",
            "--device",
            "cuda",
            "--num-workers",
            "2",
            "--skip-existing",
        ]
    )
else:
    print("Registered training is disabled. Set RUN_REGISTERED_TRAINING=True to start.")

## 8. Evaluate validation for all 12 checkpoints — long, explicit opt-in

This stage uses only `val`; it never acknowledges or reads the locked test split. Finish all model-selection and analysis decisions from these outputs before creating the freeze record.


In [ ]:
RUN_VALIDATION_EVALUATION = False

if RUN_VALIDATION_EVALUATION:
    require_cuda()
    run(
        [
            "uv",
            "run",
            "python",
            "scripts/evaluate_matrix.py",
            "--outputs-root",
            str(OUTPUTS_ROOT),
            "--split",
            "val",
            "--device",
            "cuda",
            "--num-workers",
            "2",
            "--skip-latency",
            "--skip-existing",
        ]
    )
else:
    print("Validation evaluation is disabled. It does not access the locked test split.")

## 9. Human protocol-freeze checkpoint

Inspect every validation artifact, freeze all modeling and analysis choices, and document any deviations outside this notebook. Then set `CREATE_FREEZE_RECORD=True` and type the statement exactly. The statement is intentionally blank by default. The cell requires all 12 checkpoints, all 12 validation metrics files, the validation comparison, a clean Git checkout, and the unchanged committed manifest.


In [ ]:
FREEZE_STATEMENT = "I reviewed validation and froze all model and analysis choices."
FREEZE_CONFIRMATION = ""
CREATE_FREEZE_RECORD = False
FREEZE_PATH = OUTPUTS_ROOT / "protocol_freeze.json"

if CREATE_FREEZE_RECORD:
    if FREEZE_CONFIRMATION != FREEZE_STATEMENT:
        raise PermissionError("Type the exact freeze statement before unlocking test.")
    required_validation_artifacts = []
    for configuration in REGISTERED_CONFIGURATIONS:
        for seed in REGISTERED_SEEDS:
            run_root = OUTPUTS_ROOT / configuration / f"seed_{seed}"
            required_validation_artifacts.extend(
                [run_root / "best.pt", run_root / "evaluation" / "val" / "metrics.json"]
            )
    required_validation_artifacts.append(
        GENERATED_REPORTS_ROOT / "comparison_val" / "comparison_metadata.json"
    )
    missing = [str(path) for path in required_validation_artifacts if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Validation stage is incomplete: {missing}")
    git_status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True
    ).strip()
    if git_status:
        raise RuntimeError("Commit project changes before freezing the registered protocol.")
    current_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
    ).strip()
    current_manifest_sha256 = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
    if current_manifest_sha256 != MANIFEST_SHA256:
        raise RuntimeError("Manifest changed after verification; restart the protocol review.")
    if FREEZE_PATH.exists():
        raise FileExistsError(f"Refusing to replace existing freeze record: {FREEZE_PATH}")
    freeze_record = {
        "schema_version": 1,
        "statement": FREEZE_CONFIRMATION,
        "confirmed_at_utc": datetime.now(UTC).isoformat(),
        "git_revision": current_revision,
        "manifest_sha256": current_manifest_sha256,
        "validation_artifacts_checked": len(required_validation_artifacts),
    }
    FREEZE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_freeze = FREEZE_PATH.with_suffix(".json.tmp")
    temporary_freeze.write_text(
        json.dumps(freeze_record, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    os.replace(temporary_freeze, FREEZE_PATH)
    print(f"Protocol freeze recorded at {FREEZE_PATH}")
else:
    print("No freeze record created. Locked-test execution remains disabled.")

## 10. Locked test, false-positive/error analysis, and robustness — longest opt-in stage

Run this exactly once after the human freeze. The cell verifies the freeze against the current Git revision and manifest, then passes the explicit test acknowledgement to the project runner. It evaluates all 12 checkpoints, produces error-analysis artifacts, runs the registered robustness suite, and aggregates the registered comparison. Test results must not be used to revise model or analysis choices.


In [ ]:
RUN_LOCKED_TEST = False

if RUN_LOCKED_TEST:
    if not FREEZE_PATH.is_file():
        raise PermissionError("Create and review the human protocol-freeze record first.")
    freeze_record = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))
    current_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
    ).strip()
    current_manifest_sha256 = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
    if freeze_record.get("statement") != FREEZE_STATEMENT:
        raise PermissionError("Freeze statement is absent or invalid.")
    if freeze_record.get("git_revision") != current_revision:
        raise PermissionError("Git revision changed after protocol freeze.")
    if freeze_record.get("manifest_sha256") != current_manifest_sha256:
        raise PermissionError("Manifest changed after protocol freeze.")
    require_cuda()
    run(
        [
            "uv",
            "run",
            "python",
            "scripts/evaluate_matrix.py",
            "--outputs-root",
            str(OUTPUTS_ROOT),
            "--split",
            "test",
            "--acknowledge-test-use",
            "--with-error-analysis",
            "--with-robustness",
            "--device",
            "cuda",
            "--num-workers",
            "2",
            "--skip-latency",
            "--skip-existing",
        ]
    )
else:
    print("Locked-test stage is disabled. Set RUN_LOCKED_TEST=True only after freezing.")

## 11. Locate generated evidence

This final cell only reports which artifact roots actually exist. It does not invent metric values. Download or persist both roots before the cloud runtime ends.


In [ ]:
for artifact_root in (OUTPUTS_ROOT, GENERATED_REPORTS_ROOT):
    print(f"{artifact_root}: exists={artifact_root.exists()}")
print(f"Freeze record: {FREEZE_PATH if FREEZE_PATH.exists() else 'not created'}")